四個 PMF 元素的做法：

(a) 平台內同類商品近期成功率 / (d) 市場飽和度：從 zeczec_history.db 算，把「資料期間取一年」理解為：以資料集裡最新的 開始日期 往前推 365 天當作分析窗口，窗口內同「次分類」的專案數 = 飽和度，其中成功比例 = 近期成功率。


(b) Google 趨勢：從 trend_scores.csv 讀 trend_score。但要老實說一個落差：你要的是「斜率」（成長趨勢），但我們目前存的 trend_score 是整個時間序列的平均值，不是斜率，因為之前的爬蟲程式只保留了平均分數、沒保留完整的週別數據。我會先用平均值當作近似的市場熱度代理指標，並且額外幫你在爬蟲程式裡加上斜率計算（用時間序列做線性回歸取斜率），這樣之後重新跑或補跑的資料就會有真正的斜率可用。


(c) 爆款相似度：從 zeczec_history.db 抓「達標率(%) 超過門檻」的專案當作「爆款」，去 MongoDB 撈這些專案的關鍵字/特色文字，用 embedding 模型算語意向量，再算每個專案跟爆款們的最大餘弦相似度。我會用本地端免費的 sentence-transformers 多語言模型來算 embedding（不用 API，不會有額度/被擋的問題，這點考量到我們一路上被 Gemini 額度和 Google Trends 封鎖搞得很累）。

門檻/窗口這些數字我先抓經驗值（例如爆款門檻 =達標率 ≥300%）

In [2]:
# ========================================
# Step 0：安裝套件
# ========================================
!pip install -q pymongo pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.1 MB/s eta 0:00:00


In [4]:
# ========================================
# Step 1：掛載 Drive + 基本設定
# ========================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
"""
計算 PMF（市場契合度）四項指標，合併輸出成訓練用 CSV (Google Colab 版本)
=====================================================

四項指標：
  (a) 平台內同類商品近期成功率   — 從 zeczec_history.db 算
  (b) Google 趨勢               — 從 trend_scores.csv 讀 (單純合併平均熱度)
  (c) 爆款相似度                — 從 MongoDB 抓文字描述 + 本地 embedding 模型算語意相似度
  (d) 市場飽和度                — 從 zeczec_history.db 算（跟 (a) 共用同一個近期窗口）
"""

import os
import sqlite3
from datetime import timedelta
import pandas as pd
import numpy as np
from pymongo import MongoClient
from google.colab import drive
from google.colab import userdata

# ========================================
# ⚙️ 核心設定區 (請在此調整變數)
# ========================================

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 資料夾路徑設定 (請確認你的 Google Drive 中有這個資料夾)
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data'

# 3. MongoDB 連線字串 (請換成你自己的 URI)
MONGODB_URI = userdata.get('MONGODB_URI')

# 4. ⏳ 【時間窗口變數】: 決定要擷取多久以內的資料作為「近期」標準
# - 若要擷取一年資料：設定為 365
# - 若要擷取半個月資料：設定為 15
RECENT_WINDOW_DAYS = 180

# 5. (c) 爆款門檻：達標率(%) 達到這個數字（含）以上，視為「爆款」
HIT_THRESHOLD_PERCENT = 300

# ========================================
# 檔案路徑設定
# ========================================
DB_PATH = os.path.join(BASE_DIR, 'zeczec_history.db')
TREND_CSV_PATH = os.path.join(BASE_DIR, 'trend_scores.csv')
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, 'training_dataset.csv')
EMBEDDING_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

print(f"\n📂 BASE_DIR 是否存在：{os.path.exists(BASE_DIR)}")
print(f"📂 zeczec_history.db 是否存在：{os.path.exists(DB_PATH)}")
print(f"📂 trend_scores.csv 是否存在：{os.path.exists(TREND_CSV_PATH)}")


# ========================================
# Step 1：讀取三個資料來源
# ========================================
def load_history_db():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM projects_summary", conn)
    conn.close()

    df['開始日期'] = pd.to_datetime(df['開始日期'], errors='coerce')
    df['結束日期'] = pd.to_datetime(df['結束日期'], errors='coerce')
    df['達標率(%)'] = pd.to_numeric(df['達標率(%)'], errors='coerce')

    before = len(df)
    df = df.dropna(subset=['開始日期', '達標率(%)']).reset_index(drop=True)
    print(f"📊 zeczec_history.db 讀取 {before} 筆，日期/達標率有效的 {len(df)} 筆")
    return df

def load_trend_scores():
    if not os.path.exists(TREND_CSV_PATH):
        print("⚠️ 找不到 trend_scores.csv，(b) 這項會全部是空值")
        return pd.DataFrame(columns=['project_id', 'trend_score'])
    df = pd.read_csv(TREND_CSV_PATH, encoding='utf-8-sig')
    print(f"📊 trend_scores.csv 讀取 {len(df)} 筆")
    return df

def load_mongo_projects():
    client = MongoClient(MONGODB_URI)
    collection = client['zeczec_db']['keyword_analysis']
    docs = list(collection.find({}, {
        'project_id': 1, 'product_type': 1, 'style': 1, 'target_audience': 1,
        'core_features': 1, 'trend_keywords': 1, 'Long-tail_Keywords': 1,
    }))
    print(f"📊 MongoDB 讀取 {len(docs)} 筆已分析專案")
    return docs

# ========================================
# Step 2：project_id 模糊比對
# ========================================
def build_project_id_index(project_ids):
    index = {}
    for pid in project_ids:
        parts = pid.split('/')
        if len(parts) < 5:
            continue
        _, main_cat, sub_cat, status, folder_name = parts[:5]
        code = folder_name.split('_')[0]
        index[(main_cat, sub_cat, status, code)] = pid
    return index

def match_project_id(row, index):
    key = (row['主分類'], row['次分類'], row['募資狀態'], row['專案編號'])
    return index.get(key)

# ========================================
# Step 3：(a) 近期同類成功率 + (d) 市場飽和度
# ========================================
def compute_a_and_d(df):
    window_end = df['開始日期'].max()
    window_start = window_end - timedelta(days=RECENT_WINDOW_DAYS)
    print(f"🗓️ 近期窗口：{window_start.date()} ~ {window_end.date()} (共 {RECENT_WINDOW_DAYS} 天)")

    df_window = df[(df['開始日期'] >= window_start) & (df['開始日期'] <= window_end)]

    stats = (
        df_window.groupby('次分類')
        .agg(
            saturation_count=('專案編號', 'count'),
            success_count=('募資狀態', lambda s: (s == '成功').sum()),
        )
    )
    stats['recent_success_rate'] = stats['success_count'] / stats['saturation_count']

    df = df.merge(stats[['saturation_count', 'recent_success_rate']],
                   left_on='次分類', right_index=True, how='left')

    # 排除「只有自己」造成的虛高成功率
    df.loc[df['saturation_count'] <= 1, ['recent_success_rate']] = np.nan

    print("✅ (a)(d) 計算完成")
    return df

# ========================================
# Step 4：(b) Google 趨勢 (移除斜率，純合併熱度分數)
# ========================================
def compute_b(df, trend_df, id_index):
    df['_matched_project_id'] = df.apply(lambda r: match_project_id(r, id_index), axis=1)

    # 直接將 trend_score 映射過來
    trend_score_lookup = trend_df.set_index('project_id')['trend_score'].to_dict()
    df['trend_level_raw'] = df['_matched_project_id'].map(trend_score_lookup)

    # 缺值填補：同「次分類」的中位數
    df['trend_level_filled'] = df['trend_level_raw'].fillna(
        df.groupby('次分類')['trend_level_raw'].transform('median'))

    df['trend_score_is_imputed'] = df['trend_level_raw'].isna()

    matched_level = df['trend_level_raw'].notna().sum()
    print(f"✅ (b) 計算完成，{matched_level}/{len(df)} 筆有平均熱度資料 (其餘用同類別中位數填補)")
    return df

# ========================================
# Step 5：(c) 爆款相似度
# ========================================
def build_project_text(doc):
    parts = [
        doc.get('product_type', ''),
        doc.get('style', ''),
        doc.get('target_audience', ''),
        '、'.join(doc.get('core_features', []) or []),
        '、'.join(doc.get('trend_keywords', []) or []),
        '、'.join(doc.get('Long-tail_Keywords', []) or []),
    ]
    return '。'.join(p for p in parts if p and p != '無')

def compute_c(df, mongo_docs):
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    mongo_by_id = {d['project_id']: d for d in mongo_docs}

    df['is_hit'] = df['達標率(%)'] >= HIT_THRESHOLD_PERCENT
    print(f"🎯 爆款門檻：達標率 ≥ {HIT_THRESHOLD_PERCENT}%，共 {df['is_hit'].sum()} 筆符合")

    texts, valid_idx = [], []
    for i, row in df.iterrows():
        pid = row['_matched_project_id']
        doc = mongo_by_id.get(pid)
        if doc is None:
            continue
        text = build_project_text(doc)
        if not text.strip():
            continue
        texts.append(text)
        valid_idx.append(i)

    if not texts:
        print("⚠️ 沒有任何專案有可用的文字資料，(c) 全部填 NaN")
        df['hit_similarity'] = np.nan
        return df

    print(f"🧠 載入本地 embedding 模型 {EMBEDDING_MODEL_NAME}（稍等一下）...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embeddings = model.encode(texts, show_progress_bar=False, normalize_embeddings=True)

    idx_in_valid = {row_idx: pos for pos, row_idx in enumerate(valid_idx)}
    hit_positions = [idx_in_valid[i] for i in valid_idx if df.loc[i, 'is_hit'] and i in idx_in_valid]

    df['hit_similarity'] = np.nan

    if not hit_positions:
        print("⚠️ 有文字資料的專案裡，沒有任何一筆是爆款，(c) 全部填 NaN")
        return df

    hit_embeddings = embeddings[hit_positions]
    sim_matrix = cosine_similarity(embeddings, hit_embeddings)

    for pos, row_idx in enumerate(valid_idx):
        sims = sim_matrix[pos].copy()
        if row_idx in hit_positions:
            self_pos_in_hits = hit_positions.index(row_idx) if row_idx in hit_positions else None
            if self_pos_in_hits is not None:
                sims = np.delete(sims, hit_positions.index(row_idx))
        df.loc[row_idx, 'hit_similarity'] = float(sims.max()) if len(sims) else np.nan

    matched = df['hit_similarity'].notna().sum()
    print(f"✅ (c) 計算完成，{matched}/{len(df)} 筆算出爆款相似度")
    return df

# ========================================
# Step 6：組合輸出
# ========================================
def main():
    history_df = load_history_db()
    trend_df = load_trend_scores()
    mongo_docs = load_mongo_projects()

    if history_df.empty:
        print("❌ zeczec_history.db 無有效資料，終止執行。")
        return

    all_known_ids = set(trend_df['project_id'].tolist()) | {d['project_id'] for d in mongo_docs}
    id_index = build_project_id_index(all_known_ids)

    df = history_df.copy()
    df = compute_a_and_d(df)
    df = compute_b(df, trend_df, id_index)
    df = compute_c(df, mongo_docs)

    output_columns = [
        '主分類', '次分類', '募資狀態', '專案編號', '方案價格列表', '折扣層數',
        'FAQ總題數', 'FAQ更新頻率', '開始日期', '結束日期', '總天數', '達標率(%)',
        'recent_success_rate',
        'trend_level_filled',
        'trend_score_is_imputed',
        'hit_similarity',
        'saturation_count',
        'is_hit',
        '_matched_project_id',
    ]

    output_df = df[output_columns].rename(columns={
        'recent_success_rate': 'pmf_a_recent_category_success_rate',
        'trend_level_filled': 'pmf_b_trend_level',
        'trend_score_is_imputed': 'pmf_b_is_imputed',
        'hit_similarity': 'pmf_c_hit_similarity',
        'saturation_count': 'pmf_d_market_saturation_count',
        '_matched_project_id': 'matched_full_project_id',
    })

    output_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
    print(f"\n🎉 完成！輸出到：{OUTPUT_CSV_PATH}")
    print(f"共 {len(output_df)} 筆，欄位：{list(output_df.columns)}")

if __name__ == '__main__':
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📂 BASE_DIR 是否存在：True
📂 zeczec_history.db 是否存在：True
📂 trend_scores.csv 是否存在：True
📊 zeczec_history.db 讀取 2104 筆，日期/達標率有效的 2104 筆
📊 trend_scores.csv 讀取 2118 筆
📊 MongoDB 讀取 2104 筆已分析專案
🗓️ 近期窗口：2026-01-01 ~ 2026-06-30 (共 180 天)
✅ (a)(d) 計算完成
✅ (b) 計算完成，2104/2104 筆有平均熱度資料 (其餘用同類別中位數填補)
🎯 爆款門檻：達標率 ≥ 300%，共 952 筆符合
🧠 載入本地 embedding 模型 paraphrase-multilingual-MiniLM-L12-v2（稍等一下）...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ (c) 計算完成，2104/2104 筆算出爆款相似度

🎉 完成！輸出到：/content/drive/MyDrive/ZecZec_Group_Data/pmf_training_dataset.csv
共 2104 筆，欄位：['主分類', '次分類', '募資狀態', '專案編號', '方案價格列表', '折扣層數', 'FAQ總題數', 'FAQ更新頻率', '開始日期', '結束日期', '總天數', '達標率(%)', 'pmf_a_recent_category_success_rate', 'pmf_b_trend_level', 'pmf_b_is_imputed', 'pmf_c_hit_similarity', 'pmf_d_market_saturation_count', 'is_hit', 'matched_full_project_id']
